# Reprodaction of ECOD (Li et al., 2023)


In [ ]:
#Install PyOD, the library that contains the ECOD code of the authors.
#Colab does not have it by default. The -q flag just hides the install messages.
!pip install -q pyod

In [ ]:
#numpy and pandas are used to work with the data and the results table
import numpy as np
import pandas as pd
#io and urllib are used to download the data files straight from GitHub
import io
import urllib.request
#train_test_split makes the random 60/40 split of the data
from sklearn.model_selection import train_test_
#The two scores the paper uses: ROC-AUC and average precision (AP)
from sklearn.metrics import roc_auc_score, average_precision_score
# ECOD is the method from the paper.
# IForest (Isolation Forest) is one of the methods the paper compares with.
# I only use it as a check that my data and set-up are the same as in the paper.
from pyod.models.ecod import ECOD
from pyod.models.iforest import IForest

## Settings
ECOD itself has no parameters. The values below are the settings of the experiment, taken from Section 4.1 and Table 1 of the paper.

In [ ]:
#Number of times the experiment is repeated (the paper uses 10)
N_TRIALS = 10
#Part of the data used for testing (the paper uses 60% train and 40% test)
TEST_SIZE = 0.4
#The paper uses only 10,000 random rows of the Shuttle dataset (see note under Table 1)
SHUTTLE_N = 10_000
# Web address of the folder with the data files.
# ADBench is a GitHub project that keeps copies of the ODDS and DAMI benchmark files the paper used.
# This address only works with a file name added to the end, for example 4_breastw.npz.
# To look at the folder in a browser use:
# https://github.com/Minqi824/ADBench/tree/main/adbench/datasets/Classical
BASE = "https://raw.githubusercontent.com/Minqi824/ADBench/main/adbench/datasets/Classical/"

# The 21 datasets I could find with the same size as in Table 1 of the paper.
# For each file I store: the name used in the paper, the ECOD ROC-AUC from Table 4
# and the ECOD average precision from Table 5.
DATASETS = {
    "4_breastw":          ("Breastw (mat)",           0.994, 0.988),
    "6_cardio":           ("Cardio (mat)",            0.897, 0.579),
    "18_Ionosphere":      ("Ionosphere (mat)",        0.831, 0.719),
    "21_Lymphography":    ("Lympho (mat)",            0.994, 0.894),
    "23_mammography":     ("Mammography (mat)",       0.894, 0.429),
    "26_optdigits":       ("Optdigits (mat)",         0.733, 0.053),
    "29_Pima":            ("Pima (mat)",              0.664, 0.541),
    "30_satellite":       ("Satellite (mat)",         0.661, 0.585),
    "31_satimage-2":      ("Satimage-2 (mat)",        0.985, 0.860),
    "32_shuttle":         ("Shuttle (mat)",           0.998, 0.981),
    "36_speech":          ("Speech (mat)",            0.485, 0.020),
    "45_wine":            ("Wine (mat)",              0.949, 0.608),
    "7_Cardiotocography": ("Cardiotocography (arff)", 0.637, 0.378),
    "15_Hepatitis":       ("Hepatitis (arff)",        0.843, 0.585),
    "17_InternetAds":     ("InternetAds (arff)",      0.676, 0.510),
    "35_SpamBase":        ("SpamBase (arff)",         0.701, 0.567),
    "37_Stamps":          ("Stamps (arff)",           0.939, 0.464),
    "41_Waveform":        ("Waveform (arff)",         0.787, 0.079),
    "42_WBC":             ("WBC (arff)",              0.990, 0.838),
    "43_WDBC":            ("WDBC (arff)",             0.983, 0.841),
    "46_WPBC":            ("WPBC (arff)",             0.547, 0.244),
}

## Load the data


In [ ]:
def load(name):
    #Download the file from GitHub into memory
    raw = urllib.request.urlopen(BASE + name + ".npz").read()
    #Open the .npz file (a numpy file that can hold several arrays)
    data = np.load(io.BytesIO(raw), allow_pickle=True)
    #X is the data and y is the label (1 = outlier, 0 = normal)
    X = data["X"].astype(float)
    y = data["y"].astype(int)
    #For Shuttle the paper only uses 10,000 random rows.
    #RandomState(0) fixes the random choice so the result is the same every time.
    if name == "32_shuttle":
        idx = np.random.RandomState(0).choice(len(X), SHUTTLE_N, replace=False)
        X, y = X[idx], y[idx]
    return X, y
#Quick test, load one dataset and print its size (rows, columns) and the share of outliers
X, y = load("4_breastw")
print(X.shape, round(y.mean() * 100, 2), "% outliers")

## Run the experiment

In [ ]:
#Here I collect one row of results per dataset
rows = []
for file, (name, paper_roc, paper_ap) in DATASETS.items():
    X, y = load(file)

    #Lists to keep the ROC-AUC and AP of each run, for ECOD and for Isolation Forest
    ecod_roc, ecod_ap, if_roc = [], [], []

    for seed in range(N_TRIALS):
        #Random 60/40 split. The seed (0 to 9) makes every run different but repeatable.
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=seed)

        #ECOD with default settings. Labels are not used for fitting (unsupervised).
        ecod = ECOD()
        ecod.fit(X_train)
        scores = ecod.decision_function(X_test)   #outlier score for every test point

        #Compare the scores with the true labels
        ecod_roc.append(roc_auc_score(y_test, scores))
        ecod_ap.append(average_precision_score(y_test, scores))

        #Same steps for Isolation Forest (only used as a check)
        iforest = IForest(random_state=seed)
        iforest.fit(X_train)
        if_roc.append(roc_auc_score(y_test, iforest.decision_function(X_test)))

    #Save the mean over the 10 runs next to the numbers from the paper
    rows.append({
        "Dataset": name,
        "n": len(X),
        "d": X.shape[1],
        "Paper ROC": paper_roc,
        "My ROC": np.mean(ecod_roc),
        "Paper AP": paper_ap,
        "My AP": np.mean(ecod_ap),
        "IForest ROC": np.mean(if_roc),
    })
    print("done:", name)

#Put everything in a table and add a row with the averages
results = pd.DataFrame(rows)
results["ROC diff"] = results["My ROC"] - results["Paper ROC"]
results["AP diff"] = results["My AP"] - results["Paper AP"]
results.loc[len(results)] = ["Average", "", ""] + list(results.iloc[:, 3:].mean())

# Show the table rounded to 3 decimals
results.round(3)

In [ ]:
#report table
results.to_csv("results.csv", index=False)

#The last row of the table holds the averages
avg = results.iloc[-1]
print(f"ECOD ROC-AUC: mine {avg['My ROC']:.3f}, paper {avg['Paper ROC']:.3f}")
print(f"ECOD AP:      mine {avg['My AP']:.3f}, paper {avg['Paper AP']:.3f}")
print(f"IForest ROC-AUC (check): mine {avg['IForest ROC']:.3f}, paper 0.800")